In [ ]:
# %% [markdown]
# # **Feature Engineering for Banking Fraud Detection**
# ## VeritasFinancial - Advanced Feature Development
# 
# **Objective**: Create comprehensive, domain-specific features that capture fraud patterns in banking transactions.
# 
# **Key Areas**:
# 1. Transaction-based features (amount, frequency, velocity)
# 2. Temporal features (time patterns, seasonality)
# 3. Behavioral features (customer spending patterns)
# 4. Location-based features (geographic risk)
# 5. Device fingerprinting features
# 6. Graph/Network features
# 7. Interaction features
# 8. Statistical features (rolling windows, aggregations)

# %% [markdown]
# ### **1. Environment Setup and Imports**

# %%
# Core data science libraries
import numpy as np  # Numerical computing
import pandas as pd  # Data manipulation
from datetime import datetime, timedelta  # Date/time handling
import warnings  # Suppress warnings
warnings.filterwarnings('ignore')

# Statistical libraries
from scipy import stats  # Statistical functions
from scipy.stats import skew, kurtosis  # Distribution analysis
from scipy.special import inv_boxcox  # Box-Cox transformation

# Machine learning preprocessing
from sklearn.preprocessing import (
    StandardScaler,  # Standardize features
    RobustScaler,    # Robust to outliers
    MinMaxScaler,    # Scale to [0,1]
    LabelEncoder,    # Encode categorical
    OneHotEncoder,   # One-hot encoding
    KBinsDiscretizer # Binning continuous variables
)

# Feature engineering libraries
from feature_engine import creation  # Advanced feature creation
from feature_engine.datetime import DatetimeFeatures  # DateTime features
from feature_engine.encoding import RareLabelEncoder  # Handle rare categories
from feature_engine.outliers import Winsorizer  # Handle outliers
from feature_engine.discretisation import EqualFrequencyDiscretiser  # Binning

# Time series features
from tsfresh import extract_features  # Time series feature extraction
from tsfresh.feature_extraction import EfficientFCParameters  # Efficient computation

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)

# %% [markdown]
# ### **2. Load and Prepare Data**

# %%
class DataLoader:
    """
    Comprehensive data loader with validation and preprocessing.
    
    This class handles loading of all data sources needed for feature engineering,
    including transactions, customer profiles, and device information.
    """
    
    def __init__(self, data_path='../data/processed/'):
        """
        Initialize DataLoader with path to data directory.
        
        Args:
            data_path: Path to processed data directory
        """
        self.data_path = data_path
        self.data = {}
        
    def load_all_data(self):
        """
        Load all required datasets with validation.
        
        Returns:
            Dictionary containing all loaded datasets
        """
        print("=" * 60)
        print("LOADING DATASETS FOR FEATURE ENGINEERING")
        print("=" * 60)
        
        # 1. Load transaction data (main dataset)
        print("\n📊 Loading transaction data...")
        self.data['transactions'] = self._load_transactions()
        print(f"   ✓ Transactions loaded: {len(self.data['transactions']):,} rows")
        print(f"   ✓ Features available: {self.data['transactions'].shape[1]} columns")
        
        # 2. Load customer profiles
        print("\n👤 Loading customer profiles...")
        self.data['customers'] = self._load_customers()
        print(f"   ✓ Customers loaded: {len(self.data['customers']):,} profiles")
        
        # 3. Load device information
        print("\n📱 Loading device information...")
        self.data['devices'] = self._load_devices()
        print(f"   ✓ Devices loaded: {len(self.data['devices']):,} records")
        
        # 4. Load merchant data
        print("\n🏪 Loading merchant data...")
        self.data['merchants'] = self._load_merchants()
        print(f"   ✓ Merchants loaded: {len(self.data['merchants']):,} records")
        
        # Validate data integrity
        self._validate_data()
        
        return self.data
    
    def _load_transactions(self):
        """Load transaction data with proper parsing"""
        try:
            df = pd.read_csv(f'{self.data_path}/transactions.csv', parse_dates=['transaction_time'])
            
            # Sort by time for temporal features
            df = df.sort_values(['customer_id', 'transaction_time']).reset_index(drop=True)
            
            return df
        except FileNotFoundError:
            # Generate synthetic data for demonstration
            return self._generate_synthetic_transactions()
    
    def _load_customers(self):
        """Load customer profile data"""
        try:
            return pd.read_csv(f'{self.data_path}/customers.csv')
        except FileNotFoundError:
            return self._generate_synthetic_customers()
    
    def _load_devices(self):
        """Load device fingerprinting data"""
        try:
            return pd.read_csv(f'{self.data_path}/devices.csv')
        except FileNotFoundError:
            return self._generate_synthetic_devices()
    
    def _load_merchants(self):
        """Load merchant risk data"""
        try:
            return pd.read_csv(f'{self.data_path}/merchants.csv')
        except FileNotFoundError:
            return self._generate_synthetic_merchants()
    
    def _generate_synthetic_transactions(self, n_transactions=100000):
        """
        Generate synthetic transaction data for demonstration.
        
        This creates realistic banking transaction patterns including:
        - Normal daily spending
        - Fraudulent patterns (unusual amounts, times, locations)
        - Customer behavior variations
        """
        np.random.seed(42)  # For reproducibility
        
        print("   ⚠ Generating synthetic transaction data...")
        
        # Generate time range (last 30 days)
        end_time = datetime.now()
        start_time = end_time - timedelta(days=30)
        
        # Generate transaction times
        transaction_times = [
            start_time + timedelta(
                seconds=np.random.randint(0, int((end_time - start_time).total_seconds()))
            )
            for _ in range(n_transactions)
        ]
        
        # Generate customer IDs (skewed distribution - some customers transact more)
        n_customers = 5000
        customer_probs = np.random.exponential(scale=0.5, size=n_customers)
        customer_probs = customer_probs / customer_probs.sum()
        customer_ids = np.random.choice(
            [f'CUST_{i:05d}' for i in range(n_customers)],
            size=n_transactions,
            p=customer_probs
        )
        
        # Generate transaction amounts with realistic distribution
        # Log-normal distribution for transaction amounts
        amounts = np.random.lognormal(mean=4.0, sigma=1.2, size=n_transactions)
        
        # Generate fraud labels (imbalanced - ~0.5% fraud rate)
        fraud_indices = np.random.choice(
            n_transactions,
            size=int(n_transactions * 0.005),
            replace=False
        )
        is_fraud = np.zeros(n_transactions, dtype=int)
        is_fraud[fraud_indices] = 1
        
        # Make fraud transactions have different characteristics
        amounts[fraud_indices] = amounts[fraud_indices] * np.random.uniform(2, 10, len(fraud_indices))
        
        # Generate merchant categories
        merchant_categories = [
            'Groceries', 'Restaurants', 'Shopping', 'Travel', 'Entertainment',
            'Healthcare', 'Utilities', 'Education', 'Services', 'Other'
        ]
        
        # Generate locations
        countries = ['US', 'UK', 'CA', 'AU', 'DE', 'FR', 'JP', 'SG', 'BR', 'IN']
        cities = {
            'US': ['New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix'],
            'UK': ['London', 'Manchester', 'Birmingham', 'Liverpool', 'Glasgow'],
            'CA': ['Toronto', 'Vancouver', 'Montreal', 'Calgary', 'Ottawa'],
            # Add more cities as needed
        }
        
        df = pd.DataFrame({
            'transaction_id': [f'TXN_{i:08d}' for i in range(n_transactions)],
            'customer_id': customer_ids,
            'amount': amounts,
            'currency': np.random.choice(['USD', 'EUR', 'GBP', 'JPY'], n_transactions),
            'transaction_time': transaction_times,
            'merchant_id': [f'MERCH_{np.random.randint(0, 1000):04d}' for _ in range(n_transactions)],
            'merchant_category': np.random.choice(merchant_categories, n_transactions),
            'country': np.random.choice(countries, n_transactions),
            'city': [np.random.choice(cities.get(c, ['Unknown'])) for c in np.random.choice(countries, n_transactions)],
            'device_id': [f'DEV_{np.random.randint(0, 2000):05d}' for _ in range(n_transactions)],
            'ip_address': [f'{np.random.randint(1,255)}.{np.random.randint(0,255)}.{np.random.randint(0,255)}.{np.random.randint(0,255)}' 
                          for _ in range(n_transactions)],
            'is_fraud': is_fraud
        })
        
        return df
    
    def _generate_synthetic_customers(self, n_customers=5000):
        """Generate synthetic customer profiles"""
        df = pd.DataFrame({
            'customer_id': [f'CUST_{i:05d}' for i in range(n_customers)],
            'age': np.random.randint(18, 85, n_customers),
            'income_bracket': np.random.choice(['Low', 'Medium', 'High', 'Very High'], n_customers),
            'account_age_days': np.random.randint(1, 3650, n_customers),
            'credit_score': np.random.randint(300, 850, n_customers),
            'avg_monthly_spend': np.random.lognormal(mean=7, sigma=1, size=n_customers),
            'num_previous_frauds': np.random.poisson(0.1, n_customers),
            'home_country': np.random.choice(['US', 'UK', 'CA', 'AU'], n_customers),
            'home_city': np.random.choice(['City_A', 'City_B', 'City_C'], n_customers)
        })
        return df
    
    def _generate_synthetic_devices(self, n_devices=2000):
        """Generate synthetic device fingerprint data"""
        df = pd.DataFrame({
            'device_id': [f'DEV_{i:05d}' for i in range(n_devices)],
            'device_type': np.random.choice(['Mobile', 'Tablet', 'Desktop', 'Laptop'], n_devices),
            'os': np.random.choice(['iOS', 'Android', 'Windows', 'macOS', 'Linux'], n_devices),
            'browser': np.random.choice(['Chrome', 'Safari', 'Firefox', 'Edge'], n_devices),
            'is_emulator': np.random.choice([0, 1], n_devices, p=[0.99, 0.01]),
            'is_rooted': np.random.choice([0, 1], n_devices, p=[0.95, 0.05]),
            'screen_resolution': np.random.choice(['1920x1080', '1366x768', '375x812', '414x896'], n_devices),
            'timezone': np.random.choice(['GMT-5', 'GMT+0', 'GMT+1', 'GMT+8'], n_devices)
        })
        return df
    
    def _generate_synthetic_merchants(self, n_merchants=1000):
        """Generate synthetic merchant risk data"""
        df = pd.DataFrame({
            'merchant_id': [f'MERCH_{i:04d}' for i in range(n_merchants)],
            'merchant_category': np.random.choice(
                ['Groceries', 'Restaurants', 'Shopping', 'Travel', 'Entertainment',
                 'Healthcare', 'Utilities', 'Education', 'Services', 'Other'], n_merchants
            ),
            'risk_score': np.random.beta(1, 5, n_merchants),  # Most merchants low risk
            'avg_transaction_amount': np.random.lognormal(mean=4, sigma=1, size=n_merchants),
            'chargeback_rate': np.random.beta(1, 20, n_merchants),
            'country': np.random.choice(['US', 'UK', 'CA', 'AU', 'DE', 'FR'], n_merchants)
        })
        return df
    
    def _validate_data(self):
        """Validate data integrity and relationships"""
        print("\n🔍 Validating data integrity...")
        
        # Check for missing values
        for name, df in self.data.items():
            missing = df.isnull().sum().sum()
            print(f"   {name}: {missing} missing values")
        
        # Check foreign key relationships
        if 'transactions' in self.data and 'customers' in self.data:
            txn_customers = set(self.data['transactions']['customer_id'].unique())
            profile_customers = set(self.data['customers']['customer_id'].unique())
            missing_customers = txn_customers - profile_customers
            print(f"   Transactions with missing customer profiles: {len(missing_customers)}")

# %% [markdown]
# ### **3. Transaction Amount Features**

# %%
class AmountFeatureEngineer:
    """
    Comprehensive feature engineering for transaction amounts.
    
    This class creates features related to transaction amounts including:
    - Basic transformations (log, sqrt, Box-Cox)
    - Statistical measures (z-scores, percentiles)
    - Ratio features (compared to customer averages)
    - Categorical features (amount bins)
    """
    
    def __init__(self, amount_col='amount'):
        """
        Initialize amount feature engineer.
        
        Args:
            amount_col: Name of the amount column
        """
        self.amount_col = amount_col
        self.boxcox_lambda = None  # Will be fitted for Box-Cox transformation
        
    def create_features(self, df, customer_history=None):
        """
        Create all amount-based features.
        
        Args:
            df: DataFrame containing transactions
            customer_history: Optional customer-level historical data
            
        Returns:
            DataFrame with added amount features
        """
        df = df.copy()
        
        print("\n" + "="*60)
        print("CREATING AMOUNT-BASED FEATURES")
        print("="*60)
        
        # 1. Basic transformations
        print("\n1️⃣ Basic transformations...")
        df = self._create_basic_transformations(df)
        
        # 2. Statistical features
        print("2️⃣ Statistical features...")
        df = self._create_statistical_features(df)
        
        # 3. Ratio features (compared to customer averages)
        print("3️⃣ Ratio features...")
        df = self._create_ratio_features(df, customer_history)
        
        # 4. Categorical features (binned amounts)
        print("4️⃣ Categorical features...")
        df = self._create_categorical_features(df)
        
        # 5. Outlier indicators
        print("5️⃣ Outlier indicators...")
        df = self._create_outlier_features(df)
        
        # 6. Amount volatility features
        print("6️⃣ Volatility features...")
        df = self._create_volatility_features(df)
        
        print(f"\n✅ Added {len([c for c in df.columns if c.startswith('amount_')])} amount features")
        
        return df
    
    def _create_basic_transformations(self, df):
        """
        Create basic mathematical transformations of amount.
        
        These transformations help normalize the distribution and
        make patterns more detectable by ML algorithms.
        """
        # Log transformation (handle zero amounts)
        # log1p = log(1 + x) to handle zeros
        df['amount_log'] = np.log1p(df[self.amount_col])
        print(f"   ✓ amount_log: Log transformation (skewness reduced from "
              f"{skew(df[self.amount_col]):.2f} to {skew(df['amount_log']):.2f})")
        
        # Square root transformation
        # Good for count data and reduces right skew
        df['amount_sqrt'] = np.sqrt(df[self.amount_col])
        
        # Box-Cox transformation (requires positive values)
        # Finds optimal lambda to normalize distribution
        if self.boxcox_lambda is None:
            # Fit Box-Cox on non-zero values
            positive_amounts = df[df[self.amount_col] > 0][self.amount_col]
            transformed, self.boxcox_lambda = stats.boxcox(positive_amounts)
            print(f"   ✓ Box-Cox lambda fitted: {self.boxcox_lambda:.3f}")
        
        # Apply Box-Cox transformation
        df['amount_boxcox'] = stats.boxcox(
            np.maximum(df[self.amount_col], 0.001),  # Ensure positive
            lmbda=self.boxcox_lambda
        )
        
        # Reciprocal transformation (for rates/frequencies)
        df['amount_reciprocal'] = 1 / (df[self.amount_col] + 1e-8)  # Avoid division by zero
        
        # Power transformations
        df['amount_squared'] = df[self.amount_col] ** 2
        df['amount_cubed'] = df[self.amount_col] ** 3
        
        return df
    
    def _create_statistical_features(self, df):
        """
        Create statistical features based on amount.
        
        These features measure how unusual a transaction amount is
        compared to statistical distributions.
        """
        # Z-score (standard deviation from mean)
        # Measures how many standard deviations from the mean
        mean_amount = df[self.amount_col].mean()
        std_amount = df[self.amount_col].std()
        df['amount_zscore'] = (df[self.amount_col] - mean_amount) / (std_amount + 1e-8)
        
        # Modified Z-score (using median for robustness)
        # More robust to outliers than standard z-score
        median_amount = df[self.amount_col].median()
        mad_amount = np.median(np.abs(df[self.amount_col] - median_amount))
        df['amount_modified_zscore'] = 0.6745 * (df[self.amount_col] - median_amount) / (mad_amount + 1e-8)
        
        # Percentile rank
        # Where does this amount fall in the overall distribution?
        df['amount_percentile'] = df[self.amount_col].rank(pct=True)
        
        # Quantile-based features
        quantiles = [0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
        for q in quantiles:
            threshold = df[self.amount_col].quantile(q)
            df[f'amount_above_q{int(q*100)}'] = (df[self.amount_col] > threshold).astype(int)
        
        # Statistical moments (rolling window)
        # Skewness and kurtosis in rolling windows
        for window in [5, 10, 20]:
            df[f'amount_skew_rolling_{window}'] = (
                df.groupby('customer_id')[self.amount_col]
                .transform(lambda x: x.rolling(window, min_periods=1).skew())
            )
            
            df[f'amount_kurt_rolling_{window}'] = (
                df.groupby('customer_id')[self.amount_col]
                .transform(lambda x: x.rolling(window, min_periods=1).kurt())
            )
        
        return df
    
    def _create_ratio_features(self, df, customer_history=None):
        """
        Create ratio features comparing transaction amount to customer averages.
        
        These features capture deviations from normal customer behavior.
        """
        if customer_history is None:
            # Calculate customer-level statistics from the data
            customer_stats = df.groupby('customer_id')[self.amount_col].agg([
                ('avg_amount', 'mean'),
                ('std_amount', 'std'),
                ('median_amount', 'median'),
                ('max_amount', 'max'),
                ('min_amount', 'min'),
                ('q75_amount', lambda x: x.quantile(0.75))
            ]).reset_index()
            
            # Merge back to original dataframe
            df = df.merge(customer_stats, on='customer_id', how='left')
        else:
            # Use provided customer history
            df = df.merge(customer_history, on='customer_id', how='left')
        
        # Ratio to customer average
        df['amount_to_avg_ratio'] = df[self.amount_col] / (df['avg_amount'] + 1e-8)
        
        # Ratio to customer median (more robust)
        df['amount_to_median_ratio'] = df[self.amount_col] / (df['median_amount'] + 1e-8)
        
        # Ratio to customer max
        df['amount_to_max_ratio'] = df[self.amount_col] / (df['max_amount'] + 1e-8)
        
        # Deviation from customer average (in multiples of std)
        df['amount_deviation_std'] = (df[self.amount_col] - df['avg_amount']) / (df['std_amount'] + 1e-8)
        
        # Is amount above 75th percentile of customer's history
        df['amount_above_customer_q75'] = (df[self.amount_col] > df['q75_amount']).astype(int)
        
        # Drop intermediate columns (keep only ratio features)
        ratio_cols = [c for c in df.columns if any(x in c for x in 
                     ['_ratio', '_deviation', '_above_customer'])]
        
        # Clean up - remove statistics columns if we added them
        stat_cols = ['avg_amount', 'std_amount', 'median_amount', 'max_amount', 
                     'min_amount', 'q75_amount']
        df = df.drop(columns=[c for c in stat_cols if c in df.columns])
        
        print(f"   ✓ Created {len(ratio_cols)} ratio features")
        
        return df
    
    def _create_categorical_features(self, df):
        """
        Create categorical features by binning amounts.
        
        Binning helps capture non-linear relationships and makes
        the model more robust to exact amount values.
        """
        # Equal-width binning
        df['amount_bin_equal_width'] = pd.cut(
            df[self.amount_col], 
            bins=10, 
            labels=[f'bin_{i}' for i in range(10)]
        )
        
        # Equal-frequency binning (quantile-based)
        df['amount_bin_equal_freq'] = pd.qcut(
            df[self.amount_col], 
            q=10, 
            labels=[f'qbin_{i}' for i in range(10)],
            duplicates='drop'
        )
        
        # Domain-specific bins (common in banking)
        bins = [0, 10, 50, 100, 500, 1000, 5000, 10000, 50000, float('inf')]
        labels = ['micro', 'very_small', 'small', 'medium_small', 'medium',
                  'medium_large', 'large', 'very_large', 'huge']
        df['amount_tier'] = pd.cut(df[self.amount_col], bins=bins, labels=labels)
        
        # One-hot encode categorical amount features
        amount_cat_features = ['amount_bin_equal_width', 'amount_bin_equal_freq', 'amount_tier']
        
        for col in amount_cat_features:
            if col in df.columns:
                # Create dummy variables
                dummies = pd.get_dummies(df[col], prefix=col.replace('amount_', 'amt'))
                df = pd.concat([df, dummies], axis=1)
                
                # Keep original categorical column
                # (can be used for tree-based models)
        
        print(f"   ✓ Created categorical amount features with {len(bins)-1} tiers")
        
        return df
    
    def _create_outlier_features(self, df):
        """
        Create features indicating if amount is an outlier.
        
        Multiple outlier detection methods for robust detection.
        """
        # IQR method (robust to non-normal distributions)
        Q1 = df[self.amount_col].quantile(0.25)
        Q3 = df[self.amount_col].quantile(0.75)
        IQR = Q3 - Q1
        
        df['amount_outlier_iqr'] = (
            (df[self.amount_col] < (Q1 - 1.5 * IQR)) | 
            (df[self.amount_col] > (Q3 + 1.5 * IQR))
        ).astype(int)
        
        # Extreme IQR (more strict)
        df['amount_outlier_extreme'] = (
            (df[self.amount_col] < (Q1 - 3 * IQR)) | 
            (df[self.amount_col] > (Q3 + 3 * IQR))
        ).astype(int)
        
        # Z-score method (assumes normality)
        zscore_threshold = 3
        df['amount_outlier_zscore'] = (np.abs(df['amount_zscore']) > zscore_threshold).astype(int)
        
        # Modified Z-score (robust)
        mod_zscore_threshold = 3.5
        df['amount_outlier_mod_zscore'] = (np.abs(df['amount_modified_zscore']) > mod_zscore_threshold).astype(int)
        
        # Percentile-based
        p99 = df[self.amount_col].quantile(0.99)
        p001 = df[self.amount_col].quantile(0.01)
        df['amount_outlier_percentile'] = (
            (df[self.amount_col] > p99) | 
            (df[self.amount_col] < p001)
        ).astype(int)
        
        print(f"   ✓ Created {len([c for c in df.columns if 'outlier' in c])} outlier indicators")
        
        return df
    
    def _create_volatility_features(self, df):
        """
        Create features measuring amount volatility.
        
        High volatility in amounts might indicate unusual behavior.
        """
        # Rolling coefficient of variation (std/mean)
        for window in [3, 5, 10]:
            rolling_mean = df.groupby('customer_id')[self.amount_col].transform(
                lambda x: x.rolling(window, min_periods=1).mean()
            )
            rolling_std = df.groupby('customer_id')[self.amount_col].transform(
                lambda x: x.rolling(window, min_periods=1).std()
            )
            df[f'amount_cv_rolling_{window}'] = rolling_std / (rolling_mean + 1e-8)
        
        # Absolute percent change from previous transaction
        df['amount_pct_change'] = df.groupby('customer_id')[self.amount_col].pct_change() * 100
        df['amount_abs_pct_change'] = np.abs(df['amount_pct_change'])
        
        # Ratio to previous amount
        df['amount_to_prev_ratio'] = (
            df.groupby('customer_id')[self.amount_col].shift(1) / 
            (df[self.amount_col] + 1e-8)
        )
        
        # Running sum (cumulative amount)
        df['amount_cumulative'] = df.groupby('customer_id')[self.amount_col].cumsum()
        
        return df

# %% [markdown]
# ### **4. Temporal Features**

# %%
class TemporalFeatureEngineer:
    """
    Comprehensive feature engineering for temporal patterns.
    
    Creates features capturing:
    - Time of day, week, month patterns
    - Cyclical encoding for periodic features
    - Time gaps between transactions
    - Seasonality and trends
    - Holiday and special day indicators
    """
    
    def __init__(self, time_col='transaction_time'):
        """
        Initialize temporal feature engineer.
        
        Args:
            time_col: Name of the timestamp column
        """
        self.time_col = time_col
        
    def create_features(self, df):
        """
        Create all temporal features.
        
        Args:
            df: DataFrame with timestamp column
            
        Returns:
            DataFrame with added temporal features
        """
        df = df.copy()
        
        print("\n" + "="*60)
        print("CREATING TEMPORAL FEATURES")
        print("="*60)
        
        # Ensure datetime type
        df[self.time_col] = pd.to_datetime(df[self.time_col])
        
        # 1. Basic time components
        print("\n1️⃣ Basic time components...")
        df = self._create_basic_time_components(df)
        
        # 2. Cyclical encoding
        print("2️⃣ Cyclical encoding...")
        df = self._create_cyclical_features(df)
        
        # 3. Time gap features
        print("3️⃣ Time gap features...")
        df = self._create_time_gap_features(df)
        
        # 4. Seasonal features
        print("4️⃣ Seasonal features...")
        df = self._create_seasonal_features(df)
        
        # 5. Holiday indicators
        print("5️⃣ Holiday indicators...")
        df = self._create_holiday_features(df)
        
        # 6. Business hour indicators
        print("6️⃣ Business hour indicators...")
        df = self._create_business_hours_features(df)
        
        # 7. Aggregated time features
        print("7️⃣ Aggregated time features...")
        df = self._create_aggregated_time_features(df)
        
        temporal_features = [c for c in df.columns if c.startswith(('hour_', 'day_', 'month_', 
                              'time_', 'is_', 'season'))]
        print(f"\n✅ Added {len(temporal_features)} temporal features")
        
        return df
    
    def _create_basic_time_components(self, df):
        """
        Extract basic time components from timestamp.
        
        These are the building blocks for all temporal features.
        """
        # Hour (0-23)
        df['hour'] = df[self.time_col].dt.hour
        
        # Minute (0-59)
        df['minute'] = df[self.time_col].dt.minute
        
        # Day of week (0=Monday, 6=Sunday)
        df['day_of_week'] = df[self.time_col].dt.dayofweek
        
        # Day of month (1-31)
        df['day_of_month'] = df[self.time_col].dt.day
        
        # Day of year (1-366)
        df['day_of_year'] = df[self.time_col].dt.dayofyear
        
        # Week of year (1-53)
        df['week_of_year'] = df[self.time_col].dt.isocalendar().week
        
        # Month (1-12)
        df['month'] = df[self.time_col].dt.month
        
        # Quarter (1-4)
        df['quarter'] = df[self.time_col].dt.quarter
        
        # Year
        df['year'] = df[self.time_col].dt.year
        
        # Is weekend? (Saturday or Sunday)
        df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
        
        # Is month start/end
        df['is_month_start'] = df[self.time_col].dt.is_month_start.astype(int)
        df['is_month_end'] = df[self.time_col].dt.is_month_end.astype(int)
        
        # Is quarter start/end
        df['is_quarter_start'] = df[self.time_col].dt.is_quarter_start.astype(int)
        df['is_quarter_end'] = df[self.time_col].dt.is_quarter_end.astype(int)
        
        # Is year start/end
        df['is_year_start'] = df[self.time_col].dt.is_year_start.astype(int)
        df['is_year_end'] = df[self.time_col].dt.is_year_end.astype(int)
        
        # Time of day category
        conditions = [
            (df['hour'] >= 5) & (df['hour'] < 12),   # Morning
            (df['hour'] >= 12) & (df['hour'] < 17),  # Afternoon
            (df['hour'] >= 17) & (df['hour'] < 21),  # Evening
            (df['hour'] >= 21) | (df['hour'] < 5)    # Night
        ]
        choices = ['morning', 'afternoon', 'evening', 'night']
        df['time_of_day'] = np.select(conditions, choices, default='unknown')
        
        print(f"   ✓ Created {len([c for c in df.columns if c in ['hour', 'minute', 'day_of_week', 
              'month', 'quarter', 'year', 'is_weekend']])} basic time components")
        
        return df
    
    def _create_cyclical_features(self, df):
        """
        Create cyclical encoding for periodic features.
        
        Linear representation (e.g., hour 23 to hour 0) loses cyclical nature.
        Sin/cos transformation preserves circular relationships.
        """
        # Hour of day (24-hour cycle)
        df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
        df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
        
        # Minute of hour (60-minute cycle)
        df['minute_sin'] = np.sin(2 * np.pi * df['minute'] / 60)
        df['minute_cos'] = np.cos(2 * np.pi * df['minute'] / 60)
        
        # Day of week (7-day cycle)
        df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
        df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
        
        # Month (12-month cycle)
        df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
        df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
        
        # Day of month (~30-day cycle)
        df['dom_sin'] = np.sin(2 * np.pi * df['day_of_month'] / 31)
        df['dom_cos'] = np.cos(2 * np.pi * df['day_of_month'] / 31)
        
        # Week of year (52-week cycle)
        df['woy_sin'] = np.sin(2 * np.pi * df['week_of_year'] / 52)
        df['woy_cos'] = np.cos(2 * np.pi * df['week_of_year'] / 52)
        
        print(f"   ✓ Created {len([c for c in df.columns if '_sin' in c or '_cos' in c])} cyclical features")
        
        return df
    
    def _create_time_gap_features(self, df):
        """
        Create features based on time gaps between transactions.
        
        Unusual time gaps (too short or too long) can indicate fraud.
        """
        # Sort by customer and time for proper gap calculation
        df = df.sort_values(['customer_id', self.time_col])
        
        # Time since last transaction (in seconds)
        df['time_since_last_tx'] = df.groupby('customer_id')[self.time_col].diff().dt.total_seconds()
        
        # Time until next transaction (in seconds)
        df['time_until_next_tx'] = df.groupby('customer_id')[self.time_col].diff(-1).dt.total_seconds().abs()
        
        # Handle NaN values (first transaction for customer)
        df['time_since_last_tx'] = df['time_since_last_tx'].fillna(0)
        df['time_until_next_tx'] = df['time_until_next_tx'].fillna(0)
        
        # Transform time gaps (log scale for better distribution)
        df['time_since_last_tx_log'] = np.log1p(df['time_since_last_tx'])
        df['time_until_next_tx_log'] = np.log1p(df['time_until_next_tx'])
        
        # Categorical time gaps
        # Very short (seconds), Short (minutes), Medium (hours), Long (days), Very long (weeks+)
        bins = [0, 60, 3600, 86400, 604800, float('inf')]
        labels = ['seconds', 'minutes', 'hours', 'days', 'weeks+']
        df['time_gap_category'] = pd.cut(df['time_since_last_tx'], bins=bins, labels=labels)
        
        # Statistical features of time gaps (rolling)
        for window in [3, 5, 10]:
            # Average time gap
            df[f'avg_time_gap_{window}'] = df.groupby('customer_id')['time_since_last_tx'].transform(
                lambda x: x.rolling(window, min_periods=1).mean()
            )
            
            # Std deviation of time gaps
            df[f'std_time_gap_{window}'] = df.groupby('customer_id')['time_since_last_tx'].transform(
                lambda x: x.rolling(window, min_periods=1).std()
            ).fillna(0)
        
        # Ratio of current gap to average gap
        df['time_gap_ratio'] = df['time_since_last_tx'] / (df['avg_time_gap_10'] + 1e-8)
        
        # Is this transaction unusually quick after previous?
        df['is_unusually_quick'] = (
            df['time_since_last_tx'] < df['avg_time_gap_10'] * 0.1
        ).astype(int)
        
        # Is this transaction unusually delayed?
        df['is_unusually_delayed'] = (
            df['time_since_last_tx'] > df['avg_time_gap_10'] * 10
        ).astype(int)
        
        print(f"   ✓ Created {len([c for c in df.columns if 'time_gap' in c or 'time_since' in c])} time gap features")
        
        return df
    
    def _create_seasonal_features(self, df):
        """
        Create features capturing seasonal patterns.
        
        Fraud rates often vary by season (holiday shopping, tax season, etc.)
        """
        # Extract date for seasonal calculations
        date = df[self.time_col].dt.date
        
        # Season (meteorological)
        conditions = [
            (df['month'].isin([12, 1, 2])),   # Winter
            (df['month'].isin([3, 4, 5])),    # Spring
            (df['month'].isin([6, 7, 8])),    # Summer
            (df['month'].isin([9, 10, 11]))   # Fall
        ]
        choices = ['winter', 'spring', 'summer', 'fall']
        df['season'] = np.select(conditions, choices, default='unknown')
        
        # Days to/from major holidays
        # This is simplified - in production, use holiday calendar API
        holidays = {
            'new_years': f"{df['year'].iloc[0]}-01-01",
            'christmas': f"{df['year'].iloc[0]}-12-25",
            'thanksgiving_us': f"{df['year'].iloc[0]}-11-{self._get_thanksgiving_day(df['year'].iloc[0])}",
            # Add more holidays as needed
        }
        
        for holiday_name, holiday_date in holidays.items():
            holiday_dt = pd.to_datetime(holiday_date)
            days_diff = (df[self.time_col] - holiday_dt).dt.days
            df[f'days_to_{holiday_name}'] = days_diff
            df[f'is_{holiday_name}_week'] = (np.abs(days_diff) <= 7).astype(int)
        
        # Weekend vs weekday patterns
        df['is_friday_night'] = ((df['day_of_week'] == 4) & (df['hour'] >= 18)).astype(int)
        df['is_saturday_night'] = ((df['day_of_week'] == 5) & (df['hour'] >= 18)).astype(int)
        
        # Pay period indicators (assuming bi-weekly pay)
        df['days_since_month_start'] = df['day_of_month']
        df['is_pay_week'] = (df['day_of_month'].between(25, 31) | 
                             df['day_of_month'].between(10, 15)).astype(int)
        
        print(f"   ✓ Created {len([c for c in df.columns if 'season' in c or 'holiday' in c or 'pay' in c])} seasonal features")
        
        return df
    
    def _get_thanksgiving_day(self, year):
        """Calculate US Thanksgiving day (4th Thursday of November)"""
        import calendar
        november = calendar.monthcalendar(year, 11)
        # Find the fourth Thursday
        thanksgiving = [week[3] for week in november if week[3] != 0][3]
        return thanksgiving
    
    def _create_holiday_features(self, df):
        """
        Create holiday indicators (simplified version).
        
        In production, integrate with holiday API for accurate dates.
        """
        # Major holiday periods
        holiday_periods = {
            'christmas_period': (12, 20, 12, 26),  # Dec 20-26
            'new_year_period': (12, 29, 1, 2),     # Dec 29 - Jan 2
            'thanksgiving_period': (11, 22, 11, 28), # Nov 22-28 (approximate)
            'valentines_day': (2, 14, 2, 14),
            'mothers_day': (5, 10, 5, 12),          # Approximate
            'fathers_day': (6, 15, 6, 17),          # Approximate
            'black_friday': (11, 24, 11, 30),       # Approximate
            'cyber_monday': (11, 27, 12, 3),        # Approximate
        }
        
        for period_name, (start_month, start_day, end_month, end_day) in holiday_periods.items():
            # Create date range for holiday period
            # This is simplified - in production, handle year boundaries properly
            start_date = pd.Timestamp(year=df['year'].iloc[0], month=start_month, day=start_day)
            end_date = pd.Timestamp(year=df['year'].iloc[0], month=end_month, day=end_day)
            
            if end_date < start_date:
                end_date = pd.Timestamp(year=df['year'].iloc[0] + 1, month=end_month, day=end_day)
            
            df[f'is_{period_name}'] = ((df[self.time_col] >= start_date) & 
                                        (df[self.time_col] <= end_date)).astype(int)
        
        return df
    
    def _create_business_hours_features(self, df):
        """
        Create features indicating business hours vs off-hours.
        
        Fraud often occurs during off-hours when detection is slower.
        """
        # Define business hours (9 AM - 5 PM, Monday-Friday)
        df['is_business_hours'] = (
            (df['day_of_week'] < 5) & 
            (df['hour'] >= 9) & 
            (df['hour'] < 17)
        ).astype(int)
        
        # After hours (outside business hours)
        df['is_after_hours'] = 1 - df['is_business_hours']
        
        # Late night (11 PM - 5 AM)
        df['is_late_night'] = ((df['hour'] >= 23) | (df['hour'] < 5)).astype(int)
        
        # Early morning (5 AM - 8 AM)
        df['is_early_morning'] = ((df['hour'] >= 5) & (df['hour'] < 8)).astype(int)
        
        # Lunch hour (12 PM - 2 PM)
        df['is_lunch_hour'] = ((df['hour'] >= 12) & (df['hour'] < 14)).astype(int)
        
        # Rush hour (8-9 AM, 5-6 PM)
        df['is_rush_hour'] = (
            ((df['hour'] >= 8) & (df['hour'] < 9)) |
            ((df['hour'] >= 17) & (df['hour'] < 18))
        ).astype(int)
        
        # Weekend nights
        df['is_weekend_night'] = (df['is_weekend'] & df['is_late_night']).astype(int)
        
        return df
    
    def _create_aggregated_time_features(self, df):
        """
        Create aggregated features based on time patterns.
        
        These capture customer's typical transaction times.
        """
        # For each customer, calculate their typical transaction times
        customer_time_stats = df.groupby('customer_id').agg({
            'hour': ['mean', 'std', lambda x: x.mode()[0] if len(x.mode()) > 0 else np.nan],
            'day_of_week': ['mean', 'std', lambda x: x.mode()[0] if len(x.mode()) > 0 else np.nan],
            'is_weekend': 'mean'
        }).round(2)
        
        # Flatten column names
        customer_time_stats.columns = ['_'.join(col).strip() for col in customer_time_stats.columns.values]
        customer_time_stats = customer_time_stats.reset_index()
        customer_time_stats.columns = ['customer_id', 'avg_hour', 'std_hour', 'mode_hour',
                                       'avg_dow', 'std_dow', 'mode_dow', 'weekend_tx_ratio']
        
        # Merge back to main dataframe
        df = df.merge(customer_time_stats, on='customer_id', how='left')
        
        # Deviation from typical behavior
        df['hour_deviation'] = np.abs(df['hour'] - df['avg_hour'])
        df['dow_deviation'] = np.abs(df['day_of_week'] - df['avg_dow'])
        df['is_unusual_hour'] = (df['hour_deviation'] > 2 * df['std_hour']).astype(int)
        df['is_unusual_day'] = (df['dow_deviation'] > 2 * df['std_dow']).astype(int)
        
        # Transaction at unusual time for this customer
        df['tx_at_unusual_time'] = ((df['is_unusual_hour'] == 1) | 
                                     (df['is_unusual_day'] == 1)).astype(int)
        
        return df

# %% [markdown]
# ### **5. Behavioral Features**

# %%
class BehavioralFeatureEngineer:
    """
    Comprehensive feature engineering for customer behavior patterns.
    
    Creates features capturing:
    - Spending patterns and habits
    - Transaction velocity
    - Category preferences
    - Deviation from historical behavior
    - Risk scores based on behavior
    """
    
    def __init__(self):
        """Initialize behavioral feature engineer"""
        self.behavioral_profiles = {}
        
    def create_features(self, df):
        """
        Create all behavioral features.
        
        Args:
            df: DataFrame with transaction and customer data
            
        Returns:
            DataFrame with added behavioral features
        """
        df = df.copy()
        
        print("\n" + "="*60)
        print("CREATING BEHAVIORAL FEATURES")
        print("="*60)
        
        # 1. Spending velocity features
        print("\n1️⃣ Spending velocity features...")
        df = self._create_velocity_features(df)
        
        # 2. Category preference features
        print("2️⃣ Category preference features...")
        df = self._create_category_features(df)
        
        # 3. Deviation features
        print("3️⃣ Deviation features...")
        df = self._create_deviation_features(df)
        
        # 4. Pattern features
        print("4️⃣ Pattern features...")
        df = self._create_pattern_features(df)
        
        # 5. Risk score features
        print("5️⃣ Risk score features...")
        df = self._create_risk_features(df)
        
        # 6. Sequence features
        print("6️⃣ Sequence features...")
        df = self._create_sequence_features(df)
        
        behavioral_features = [c for c in df.columns if any(x in c for x in 
                              ['velocity', 'spend', 'category', 'deviation', 'pattern', 'risk_'])]
        print(f"\n✅ Added {len(behavioral_features)} behavioral features")
        
        return df
    
    def _create_velocity_features(self, df):
        """
        Create transaction velocity features.
        
        Velocity measures how quickly transactions occur, which is
        a strong indicator of fraud (e.g., card testing).
        """
        # Sort for rolling calculations
        df = df.sort_values(['customer_id', 'transaction_time'])
        
        # Transaction count in various windows
        windows = {
            '1H': 3600,      # 1 hour in seconds
            '3H': 10800,     # 3 hours
            '6H': 21600,     # 6 hours
            '12H': 43200,    # 12 hours
            '24H': 86400,    # 24 hours
            '7D': 604800,    # 7 days
        }
        
        for window_name, window_seconds in windows.items():
            # Count transactions in window
            df[f'tx_count_{window_name}'] = df.groupby('customer_id').rolling(
                f'{window_name}', on='transaction_time'
            )['transaction_id'].count().reset_index(0, drop=True)
            
            # Total amount spent in window
            df[f'total_spent_{window_name}'] = df.groupby('customer_id').rolling(
                f'{window_name}', on='transaction_time'
            )['amount'].sum().reset_index(0, drop=True)
            
            # Average amount in window
            df[f'avg_amount_{window_name}'] = (
                df[f'total_spent_{window_name}'] / (df[f'tx_count_{window_name}'] + 1e-8)
            )
            
            # Maximum amount in window
            df[f'max_amount_{window_name}'] = df.groupby('customer_id').rolling(
                f'{window_name}', on='transaction_time'
            )['amount'].max().reset_index(0, drop=True)
        
        # Fill NaN values (first transactions)
        velocity_cols = [c for c in df.columns if any(x in c for x in ['tx_count', 'total_spent', 'avg_amount', 'max_amount'])]
        df[velocity_cols] = df[velocity_cols].fillna(0)
        
        # Spending rate (amount per unit time)
        df['spending_rate_hourly'] = df['total_spent_1H'] / 3600  # per second
        df['spending_rate_daily'] = df['total_spent_24H'] / 86400  # per second
        
        # Velocity changes (acceleration)
        df['tx_count_accel'] = df.groupby('customer_id')['tx_count_1H'].diff().fillna(0)
        df['spending_accel'] = df.groupby('customer_id')['total_spent_1H'].diff().fillna(0)
        
        # Unusual velocity flags
        # If transaction count in last hour > 3x average
        df['avg_tx_count_7D'] = df.groupby('customer_id')['tx_count_24H'].transform(
            lambda x: x.rolling(7, min_periods=1).mean()
        )
        df['high_velocity_flag'] = (
            df['tx_count_1H'] > 3 * df['avg_tx_count_7D']
        ).astype(int)
        
        print(f"   ✓ Created {len([c for c in df.columns if 'tx_count' in c or 'spent' in c])} velocity features")
        
        return df
    
    def _create_category_features(self, df):
        """
        Create features based on merchant category preferences.
        
        Customers have typical categories they transact in.
        Deviation from these preferences may indicate fraud.
        """
        # Category distribution for each customer
        category_stats = df.groupby(['customer_id', 'merchant_category']).agg({
            'amount': ['count', 'sum', 'mean'],
            'transaction_id': 'count'
        }).round(2)
        
        category_stats.columns = ['_'.join(col).strip() for col in category_stats.columns.values]
        category_stats = category_stats.reset_index()
        
        # Pivot to get features per category
        for category in df['merchant_category'].unique():
            cat_data = category_stats[category_stats['merchant_category'] == category]
            
            if len(cat_data) > 0:
                # Number of transactions in this category
                cat_count = cat_data.set_index('customer_id')['transaction_id_count']
                df[f'tx_count_cat_{category}'] = df['customer_id'].map(cat_count).fillna(0)
                
                # Total spent in this category
                cat_total = cat_data.set_index('customer_id')['amount_sum']
                df[f'total_spent_cat_{category}'] = df['customer_id'].map(cat_total).fillna(0)
                
                # Percentage of transactions in this category
                total_tx = df.groupby('customer_id')['transaction_id'].count()
                df[f'pct_tx_cat_{category}'] = (
                    df[f'tx_count_cat_{category}'] / df['customer_id'].map(total_tx) * 100
                ).fillna(0)
        
        # Favorite category (most frequent)
        favorite_cat = df.groupby('customer_id')['merchant_category'].agg(
            lambda x: x.mode()[0] if len(x.mode()) > 0 else 'Unknown'
        ).to_dict()
        df['favorite_category'] = df['customer_id'].map(favorite_cat)
        
        # Is current transaction in favorite category?
        df['is_favorite_category'] = (df['merchant_category'] == df['favorite_category']).astype(int)
        
        # Category switching behavior
        df['category_changed'] = (df.groupby('customer_id')['merchant_category'].shift() != 
                                   df['merchant_category']).astype(int)
        
        # Number of unique categories used
        df['unique_categories_count'] = df.groupby('customer_id')['merchant_category'].transform('nunique')
        
        # Category entropy (diversity of spending)
        from scipy.stats import entropy
        
        def calculate_entropy(group):
            counts = group.value_counts(normalize=True)
            return entropy(counts)
        
        category_entropy = df.groupby('customer_id')['merchant_category'].apply(calculate_entropy)
        df['category_entropy'] = df['customer_id'].map(category_entropy)
        
        print(f"   ✓ Created {len([c for c in df.columns if 'cat_' in c or 'category' in c])} category features")
        
        return df
    
    def _create_deviation_features(self, df):
        """
        Create features measuring deviation from historical behavior.
        
        Large deviations from normal patterns are strong fraud indicators.
        """
        # Historical averages for each customer
        historical_stats = df.groupby('customer_id').agg({
            'amount': ['mean', 'std', 'median', 'skew'],
            'hour': 'mean',
            'day_of_week': 'mean',
        })
        
        historical_stats.columns = ['_'.join(col).strip() for col in historical_stats.columns.values]
        historical_stats = historical_stats.reset_index()
        
        # Merge historical stats
        df = df.merge(historical_stats, on='customer_id', how='left', suffixes=('', '_hist'))
        
        # Amount deviation
        df['amount_deviation_from_mean'] = np.abs(df['amount'] - df['amount_mean']) / (df['amount_std'] + 1e-8)
        df['amount_deviation_from_median'] = np.abs(df['amount'] - df['amount_median']) / (df['amount_median'] + 1e-8)
        
        # Time deviation
        df['hour_deviation_from_mean'] = np.abs(df['hour'] - df['hour_mean'])
        df['dow_deviation_from_mean'] = np.abs(df['day_of_week'] - df['day_of_week_mean'])
        
        # Z-score for amount (customer-level)
        df['amount_zscore_customer'] = (df['amount'] - df['amount_mean']) / (df['amount_std'] + 1e-8)
        
        # Rolling deviation (last N transactions)
        for window in [3, 5, 10]:
            rolling_mean = df.groupby('customer_id')['amount'].transform(
                lambda x: x.rolling(window, min_periods=1).mean()
            )
            rolling_std = df.groupby('customer_id')['amount'].transform(
                lambda x: x.rolling(window, min_periods=1).std()
            )
            df[f'amount_deviation_rolling_{window}'] = (df['amount'] - rolling_mean) / (rolling_std + 1e-8)
        
        # Is amount > 2 standard deviations from customer mean?
        df['is_amount_outlier_customer'] = (np.abs(df['amount_zscore_customer']) > 2).astype(int)
        
        # Is amount > 3 standard deviations from customer mean?
        df['is_amount_extreme_customer'] = (np.abs(df['amount_zscore_customer']) > 3).astype(int)
        
        # Percentage deviation
        df['amount_pct_deviation'] = (df['amount'] - df['amount_mean']) / (df['amount_mean'] + 1e-8) * 100
        
        print(f"   ✓ Created {len([c for c in df.columns if 'deviation' in c or 'zscore' in c])} deviation features")
        
        return df
    
    def _create_pattern_features(self, df):
        """
        Create features capturing behavioral patterns.
        
        Identifies recurring patterns in customer behavior.
        """
        # Regular spending patterns
        # Does customer have regular spending intervals?
        df['regular_interval_flag'] = self._detect_regular_intervals(df)
        
        # Round amount patterns (fraud often uses round amounts)
        df['is_round_amount'] = (df['amount'] % 1 == 0).astype(int)
        df['is_very_round'] = (df['amount'] % 100 == 0).astype(int)
        
        # Amount ending patterns
        df['amount_last_digit'] = (df['amount'] * 100 % 10).astype(int)
        df['is_suspicious_ending'] = df['amount_last_digit'].isin([0, 5, 9]).astype(int)
        
        # Time patterns
        # Does customer transact at same time each day/week?
        df['hour_regularity'] = df.groupby('customer_id')['hour'].transform('std')
        df['is_regular_time'] = (df['hour_regularity'] < 2).astype(int)
        
        # Weekend vs weekday pattern
        df['weekend_ratio'] = df.groupby('customer_id')['is_weekend'].transform('mean')
        
        # Night transaction pattern
        df['night_tx_ratio'] = df.groupby('customer_id')['is_late_night'].transform('mean')
        
        # Pattern change detection
        # Has the customer's pattern changed recently?
        recent_period = df['transaction_time'] > df['transaction_time'].max() - pd.Timedelta(days=7)
        if recent_period.any():
            recent_avg_amount = df[recent_period].groupby('customer_id')['amount'].mean()
            historical_avg_amount = df[~recent_period].groupby('customer_id')['amount'].mean()
            
            df['recent_amount_change'] = (
                df['customer_id'].map(recent_avg_amount) - df['customer_id'].map(historical_avg_amount)
            ) / (df['customer_id'].map(historical_avg_amount) + 1e-8)
        else:
            df['recent_amount_change'] = 0
        
        return df
    
    def _detect_regular_intervals(self, df):
        """Detect if customer has regular transaction intervals"""
        def check_regularity(group):
            if len(group) < 5:
                return 0
            
            # Calculate time differences
            time_diffs = group.diff().dt.total_seconds().dropna()
            
            if len(time_diffs) < 3:
                return 0
            
            # Check if intervals are regular (low coefficient of variation)
            cv = time_diffs.std() / (time_diffs.mean() + 1e-8)
            return 1 if cv < 0.3 else 0
        
        regularity = df.groupby('customer_id')['transaction_time'].apply(check_regularity)
        return df['customer_id'].map(regularity)
    
    def _create_risk_features(self, df):
        """
        Create risk scores based on behavioral patterns.
        
        Combine multiple indicators into composite risk scores.
        """
        # Amount risk
        df['amount_risk'] = (
            (df['amount'] > 5000).astype(int) * 3 +
            (df['amount'] > 10000).astype(int) * 2 +
            (df['amount'] > 50000).astype(int) * 5 +
            df['is_amount_outlier_customer'] * 4 +
            df['is_amount_extreme_customer'] * 3
        )
        
        # Velocity risk
        df['velocity_risk'] = (
            (df['tx_count_1H'] > 5).astype(int) * 3 +
            (df['tx_count_1H'] > 10).astype(int) * 4 +
            df['high_velocity_flag'] * 5 +
            (df['spending_rate_hourly'] > 1000).astype(int) * 3
        )
        
        # Time risk
        df['time_risk'] = (
            df['is_late_night'] * 3 +
            df['is_after_hours'] * 2 +
            df['is_unusual_hour'] * 4 +
            df['is_unusual_day'] * 3 +
            df['tx_at_unusual_time'] * 5
        )
        
        # Category risk
        df['category_risk'] = (
            (1 - df['is_favorite_category']) * 3 +
            df['category_changed'] * 2 +
            (df['category_entropy'] > 2).astype(int) * 2
        )
        
        # Combined risk score (weighted sum)
        df['behavioral_risk_score'] = (
            df['amount_risk'] * 0.3 +
            df['velocity_risk'] * 0.3 +
            df['time_risk'] * 0.2 +
            df['category_risk'] * 0.2
        )
        
        # Normalize to 0-100 scale
        min_risk = df['behavioral_risk_score'].min()
        max_risk = df['behavioral_risk_score'].max()
        df['behavioral_risk_score_norm'] = (
            (df['behavioral_risk_score'] - min_risk) / (max_risk - min_risk + 1e-8) * 100
        )
        
        # Risk tiers
        conditions = [
            df['behavioral_risk_score_norm'] < 20,
            df['behavioral_risk_score_norm'] < 40,
            df['behavioral_risk_score_norm'] < 60,
            df['behavioral_risk_score_norm'] < 80,
            df['behavioral_risk_score_norm'] >= 80
        ]
        choices = ['very_low', 'low', 'medium', 'high', 'very_high']
        df['risk_tier'] = np.select(conditions, choices, default='unknown')
        
        print(f"   ✓ Created {len([c for c in df.columns if 'risk' in c])} risk features")
        
        return df
    
    def _create_sequence_features(self, df):
        """
        Create features based on transaction sequences.
        
        The order and pattern of transactions can indicate fraud.
        """
        # Sort for sequence analysis
        df = df.sort_values(['customer_id', 'transaction_time'])
        
        # Amount trend (increasing/decreasing)
        df['amount_trend'] = df.groupby('customer_id')['amount'].transform(
            lambda x: x.rolling(5, min_periods=1).apply(
                lambda y: np.polyfit(range(len(y)), y, 1)[0] if len(y) > 1 else 0
            )
        )
        
        # Alternating pattern detection (e.g., small-large-small)
        def detect_alternating(amounts):
            if len(amounts) < 3:
                return 0
            signs = np.sign(np.diff(amounts))
            return 1 if len(set(signs)) == 1 else 0
        
        df['alternating_pattern'] = df.groupby('customer_id')['amount'].transform(
            lambda x: x.rolling(5, min_periods=3).apply(detect_alternating)
        )
        
        # Repetitive amounts
        def count_repetitions(amounts):
            if len(amounts) < 2:
                return 0
            return (amounts.iloc[-1] == amounts.iloc[-2]).astype(int)
        
        df['repetitive_amount'] = df.groupby('customer_id')['amount'].transform(
            lambda x: x.rolling(2).apply(count_repetitions)
        )
        
        # Amount monotonicity (strictly increasing/decreasing)
        def check_monotonic(amounts):
            if len(amounts) < 3:
                return 0
            return 1 if (np.all(np.diff(amounts) >= 0) or np.all(np.diff(amounts) <= 0)) else 0
        
        df['monotonic_sequence'] = df.groupby('customer_id')['amount'].transform(
            lambda x: x.rolling(5, min_periods=3).apply(check_monotonic)
        )
        
        # Gap between consecutive amounts
        df['amount_gap'] = df.groupby('customer_id')['amount'].diff().abs()
        df['amount_gap_ratio'] = df['amount_gap'] / (df['amount'] + 1e-8)
        
        return df

# %% [markdown]
# ### **6. Location-Based Features**

# %%
class LocationFeatureEngineer:
    """
    Comprehensive feature engineering for location-based risk.
    
    Creates features capturing:
    - Geographic distance and velocity
    - Country/city risk scores
    - Location anomalies
    - IP address risk
    """
    
    def __init__(self):
        """Initialize location feature engineer"""
        # Load country risk scores (simplified)
        self.country_risk = {
            'US': 0.2, 'UK': 0.2, 'CA': 0.2, 'AU': 0.2,
            'DE': 0.3, 'FR': 0.3, 'JP': 0.3, 'SG': 0.4,
            'BR': 0.6, 'IN': 0.5, 'RU': 0.7, 'CN': 0.7,
            'NG': 0.8, 'KE': 0.7, 'ZA': 0.6, 'Unknown': 0.5
        }
        
        # City risk scores (simplified)
        self.city_risk = {
            'New York': 0.3, 'Los Angeles': 0.3, 'Chicago': 0.4,
            'London': 0.2, 'Manchester': 0.3, 'Toronto': 0.2,
            'Unknown': 0.5
        }
        
    def create_features(self, df):
        """
        Create all location-based features.
        
        Args:
            df: DataFrame with location data
            
        Returns:
            DataFrame with added location features
        """
        df = df.copy()
        
        print("\n" + "="*60)
        print("CREATING LOCATION-BASED FEATURES")
        print("="*60)
        
        # 1. Basic location features
        print("\n1️⃣ Basic location features...")
        df = self._create_basic_location_features(df)
        
        # 2. Distance features
        print("2️⃣ Distance features...")
        df = self._create_distance_features(df)
        
        # 3. Location velocity
        print("3️⃣ Location velocity...")
        df = self._create_location_velocity_features(df)
        
        # 4. Risk scores
        print("4️⃣ Risk scores...")
        df = self._create_location_risk_features(df)
        
        # 5. Anomaly indicators
        print("5️⃣ Anomaly indicators...")
        df = self._create_location_anomaly_features(df)
        
        # 6. IP-based features
        print("6️⃣ IP-based features...")
        df = self._create_ip_features(df)
        
        location_features = [c for c in df.columns if any(x in c for x in 
                            ['country', 'city', 'distance', 'location', 'ip_'])]
        print(f"\n✅ Added {len(location_features)} location features")
        
        return df
    
    def _create_basic_location_features(self, df):
        """
        Create basic location indicators.
        """
        # Country risk score
        df['country_risk_score'] = df['country'].map(self.country_risk).fillna(0.5)
        
        # City risk score
        df['city_risk_score'] = df['city'].map(self.city_risk).fillna(0.5)
        
        # Is foreign transaction (different from customer's home country)
        if 'home_country' in df.columns:
            df['is_foreign_transaction'] = (df['country'] != df['home_country']).astype(int)
        else:
            df['is_foreign_transaction'] = 0
        
        # Is transaction in customer's home city
        if 'home_city' in df.columns:
            df['is_home_city'] = (df['city'] == df['home_city']).astype(int)
        else:
            df['is_home_city'] = 1
        
        # Continent mapping (simplified)
        continent_map = {
            'US': 'NA', 'CA': 'NA', 'MX': 'NA',
            'UK': 'EU', 'DE': 'EU', 'FR': 'EU', 'IT': 'EU', 'ES': 'EU',
            'CN': 'AS', 'JP': 'AS', 'IN': 'AS', 'SG': 'AS',
            'BR': 'SA', 'AR': 'SA',
            'AU': 'OC',
            'ZA': 'AF', 'NG': 'AF', 'KE': 'AF',
        }
        df['continent'] = df['country'].map(continent_map).fillna('Unknown')
        
        return df
    
    def _create_distance_features(self, df):
        """
        Create features based on geographic distance.
        
        Large distances between consecutive transactions in short time
        are impossible for normal travel and indicate fraud.
        """
        # Sort for location change analysis
        df = df.sort_values(['customer_id', 'transaction_time'])
        
        # Approximate coordinates (simplified - in production use geocoding)
        # This is a placeholder - in reality, you'd have lat/lon for each city
        city_coords = {
            'New York': (40.71, -74.01),
            'Los Angeles': (34.05, -118.24),
            'Chicago': (41.88, -87.63),
            'London': (51.51, -0.13),
            'Manchester': (53.48, -2.24),
            'Toronto': (43.65, -79.38),
            'Unknown': (0, 0)
        }
        
        # Add approximate coordinates
        df['lat'] = df['city'].map(lambda x: city_coords.get(x, (0, 0))[0])
        df['lon'] = df['city'].map(lambda x: city_coords.get(x, (0, 0))[1])
        
        # Calculate distance from previous transaction
        def haversine_distance(lat1, lon1, lat2, lon2):
            """Calculate great-circle distance between two points"""
            R = 6371  # Earth's radius in kilometers
            
            lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
            dlat = lat2 - lat1
            dlon = lon2 - lon1
            
            a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
            c = 2 * np.arcsin(np.sqrt(a))
            
            return R * c
        
        # Calculate distance to previous transaction
        prev_lat = df.groupby('customer_id')['lat'].shift()
        prev_lon = df.groupby('customer_id')['lon'].shift()
        
        df['distance_from_prev_km'] = df.apply(
            lambda row: haversine_distance(
                row['lat'], row['lon'],
                prev_lat[row.name] if pd.notna(prev_lat[row.name]) else row['lat'],
                prev_lon[row.name] if pd.notna(prev_lon[row.name]) else row['lon']
            ) if pd.notna(prev_lat[row.name]) else 0,
            axis=1
        )
        
        # Distance from home location
        if 'home_city' in df.columns:
            home_lat = df['home_city'].map(lambda x: city_coords.get(x, (0, 0))[0])
            home_lon = df['home_city'].map(lambda x: city_coords.get(x, (0, 0))[1])
            
            df['distance_from_home_km'] = df.apply(
                lambda row: haversine_distance(
                    row['lat'], row['lon'],
                    home_lat[row.name], home_lon[row.name]
                ) if pd.notna(home_lat[row.name]) else 0,
                axis=1
            )
        else:
            df['distance_from_home_km'] = 0
        
        # Log transform distances
        df['distance_from_prev_km_log'] = np.log1p(df['distance_from_prev_km'])
        df['distance_from_home_km_log'] = np.log1p(df['distance_from_home_km'])
        
        return df
    
    def _create_location_velocity_features(self, df):
        """
        Create features measuring how fast location changes.
        
        Location velocity = distance / time
        Impossible velocities indicate fraud (e.g., using VPN/proxy).
        """
        # Time since last transaction (in hours)
        time_since_prev = df['time_since_last_tx'] / 3600  # convert to hours
        
        # Location velocity (km/h)
        df['location_velocity_kmh'] = df['distance_from_prev_km'] / (time_since_prev + 1e-8)
        
        # Impossible velocity flag (> 1000 km/h is impossible for normal travel)
        df['impossible_velocity_flag'] = (df['location_velocity_kmh'] > 1000).astype(int)
        
        # Suspicious velocity (> 500 km/h but < 1000 km/h - possible but unusual)
        df['suspicious_velocity_flag'] = (
            (df['location_velocity_kmh'] > 500) & 
            (df['location_velocity_kmh'] <= 1000)
        ).astype(int)
        
        # Location change indicators
        df['location_changed'] = (df['distance_from_prev_km'] > 1).astype(int)
        df['country_changed'] = (df.groupby('customer_id')['country'].shift() != df['country']).astype(int)
        df['city_changed'] = (df.groupby('customer_id')['city'].shift() != df['city']).astype(int)
        
        return df
    
    def _create_location_risk_features(self, df):
        """
        Create composite location risk scores.
        """
        # High-risk country
        df['is_high_risk_country'] = (df['country_risk_score'] > 0.6).astype(int)
        
        # High-risk city
        df['is_high_risk_city'] = (df['city_risk_score'] > 0.6).astype(int)
        
        # Multiple countries in short time
        df['unique_countries_last_day'] = df.groupby('customer_id').rolling(
            '24H', on='transaction_time'
        )['country'].apply(lambda x: x.nunique()).reset_index(0, drop=True)
        
        # Multiple cities in short time
        df['unique_cities_last_day'] = df.groupby('customer_id').rolling(
            '24H', on='transaction_time'
        )['city'].apply(lambda x: x.nunique()).reset_index(0, drop=True)
        
        # Fill NaN
        df['unique_countries_last_day'] = df['unique_countries_last_day'].fillna(1)
        df['unique_cities_last_day'] = df['unique_cities_last_day'].fillna(1)
        
        # Location risk score (composite)
        df['location_risk_score'] = (
            df['country_risk_score'] * 0.3 +
            df['city_risk_score'] * 0.2 +
            df['is_foreign_transaction'] * 0.2 +
            df['impossible_velocity_flag'] * 0.2 +
            (df['unique_countries_last_day'] > 2).astype(int) * 0.1
        )
        
        return df
    
    def _create_location_anomaly_features(self, df):
        """
        Create features indicating location anomalies.
        """
        # Is this location unusual for the customer?
        # Most common location for customer
        most_common_country = df.groupby('customer_id')['country'].agg(
            lambda x: x.mode()[0] if len(x.mode()) > 0 else 'Unknown'
        ).to_dict()
        df['most_common_country'] = df['customer_id'].map(most_common_country)
        
        most_common_city = df.groupby('customer_id')['city'].agg(
            lambda x: x.mode()[0] if len(x.mode()) > 0 else 'Unknown'
        ).to_dict()
        df['most_common_city'] = df['customer_id'].map(most_common_city)
        
        # Is current location unusual?
        df['is_unusual_country'] = (df['country'] != df['most_common_country']).astype(int)
        df['is_unusual_city'] = (df['city'] != df['most_common_city']).astype(int)
        
        # First time in this country/city
        df['first_time_country'] = (df.groupby('customer_id')['country'].transform(
            lambda x: (x.shift() != x).cumsum()
        ) == 1).astype(int)
        
        df['first_time_city'] = (df.groupby('customer_id')['city'].transform(
            lambda x: (x.shift() != x).cumsum()
        ) == 1).astype(int)
        
        return df
    
    def _create_ip_features(self, df):
        """
        Create features based on IP address.
        
        IP addresses can reveal VPN/proxy usage and location mismatches.
        """
        # Extract IP components
        df['ip_first_octet'] = df['ip_address'].str.split('.').str[0].astype(float)
        df['ip_second_octet'] = df['ip_address'].str.split('.').str[1].astype(float)
        
        # Known VPN/proxy IP ranges (simplified)
        vpn_ranges = [
            (10, 0, 0, 0, 10, 255, 255, 255),      # 10.0.0.0/8 (private)
            (172, 16, 0, 0, 172, 31, 255, 255),    # 172.16.0.0/12 (private)
            (192, 168, 0, 0, 192, 168, 255, 255),  # 192.168.0.0/16 (private)
        ]
        
        def is_vpn_ip(row):
            ip_parts = [row['ip_first_octet'], row['ip_second_octet']]
            for start1, start2, _, _, end1, end2, _, _ in vpn_ranges:
                if start1 <= ip_parts[0] <= end1 and start2 <= ip_parts[1] <= end2:
                    return 1
            return 0
        
        df['is_vpn_ip'] = df.apply(is_vpn_ip, axis=1)
        
        # IP location mismatch
        # In production, you'd have IP geolocation data
        # Here we'll create a synthetic mismatch indicator
        df['ip_country_mismatch'] = np.random.choice([0, 1], size=len(df), p=[0.95, 0.05])
        
        # IP risk score (simplified)
        df['ip_risk_score'] = (
            df['is_vpn_ip'] * 0.5 +
            df['ip_country_mismatch'] * 0.5
        )
        
        return df

# %% [markdown]
# ### **7. Main Feature Engineering Pipeline**

# %%
class FeatureEngineeringPipeline:
    """
    Main pipeline orchestrating all feature engineering steps.
    
    This class coordinates the creation of all feature types
    and ensures proper ordering and dependencies.
    """
    
    def __init__(self, config=None):
        """
        Initialize feature engineering pipeline.
        
        Args:
            config: Configuration dictionary for feature engineering
        """
        self.config = config or self._default_config()
        self.feature_engineers = {}
        self.feature_names = []
        
    def _default_config(self):
        """Default configuration for feature engineering"""
        return {
            'amount_features': True,
            'temporal_features': True,
            'behavioral_features': True,
            'location_features': True,
            'interaction_features': True,
            'save_intermediate': False,
            'parallel_processing': False,
            'n_jobs': -1
        }
    
    def build_pipeline(self):
        """Initialize all feature engineers"""
        print("\n" + "="*60)
        print("BUILDING FEATURE ENGINEERING PIPELINE")
        print("="*60)
        
        if self.config['amount_features']:
            self.feature_engineers['amount'] = AmountFeatureEngineer()
            print("✓ Amount feature engineer initialized")
            
        if self.config['temporal_features']:
            self.feature_engineers['temporal'] = TemporalFeatureEngineer()
            print("✓ Temporal feature engineer initialized")
            
        if self.config['behavioral_features']:
            self.feature_engineers['behavioral'] = BehavioralFeatureEngineer()
            print("✓ Behavioral feature engineer initialized")
            
        if self.config['location_features']:
            self.feature_engineers['location'] = LocationFeatureEngineer()
            print("✓ Location feature engineer initialized")
        
        return self
    
    def fit_transform(self, df):
        """
        Fit and transform all features.
        
        Args:
            df: Input DataFrame
            
        Returns:
            DataFrame with all engineered features
        """
        print("\n" + "="*60)
        print("RUNNING FEATURE ENGINEERING PIPELINE")
        print("="*60)
        
        result_df = df.copy()
        initial_cols = set(result_df.columns)
        
        # Apply each feature engineer in sequence
        for name, engineer in self.feature_engineers.items():
            print(f"\n📊 Applying {name} feature engineering...")
            result_df = engineer.create_features(result_df)
            
            if self.config['save_intermediate']:
                self._save_intermediate(result_df, name)
        
        # Track new features
        final_cols = set(result_df.columns)
        new_features = final_cols - initial_cols
        self.feature_names = list(new_features)
        
        print("\n" + "="*60)
        print(f"FEATURE ENGINEERING COMPLETE")
        print("="*60)
        print(f"Original features: {len(initial_cols)}")
        print(f"New features created: {len(new_features)}")
        print(f"Total features: {len(final_cols)}")
        
        # Display feature categories
        self._summarize_features(new_features)
        
        return result_df
    
    def _save_intermediate(self, df, stage_name):
        """Save intermediate results"""
        output_path = f'../data/features/intermediate_{stage_name}.parquet'
        df.to_parquet(output_path)
        print(f"   💾 Saved intermediate results to {output_path}")
    
    def _summarize_features(self, features):
        """Summarize created features by category"""
        categories = {
            'amount': [f for f in features if 'amount' in f],
            'temporal': [f for f in features if any(x in f for x in ['hour', 'day', 'month', 'time', 'season'])],
            'behavioral': [f for f in features if any(x in f for x in ['velocity', 'spend', 'category', 'deviation', 'risk'])],
            'location': [f for f in features if any(x in f for x in ['country', 'city', 'distance', 'ip_'])],
        }
        
        print("\n📊 Feature Summary by Category:")
        for category, cat_features in categories.items():
            print(f"   {category.capitalize()}: {len(cat_features)} features")
    
    def get_feature_importance_ranking(self, df, target_col='is_fraud'):
        """
        Calculate initial feature importance using correlation with target.
        
        This provides a rough ranking of feature importance before modeling.
        """
        if target_col not in df.columns:
            print("Target column not found for importance calculation")
            return {}
        
        correlations = {}
        for feature in self.feature_names:
            if feature in df.columns and df[feature].dtype in ['int64', 'float64']:
                corr = df[feature].corr(df[target_col])
                correlations[feature] = abs(corr) if pd.notna(corr) else 0
        
        # Sort by absolute correlation
        sorted_features = sorted(correlations.items(), key=lambda x: x[1], reverse=True)
        
        print("\n📈 Top 20 Features by Correlation with Target:")
        for i, (feature, corr) in enumerate(sorted_features[:20]):
            print(f"   {i+1:2d}. {feature[:40]:40} {corr:.4f}")
        
        return dict(sorted_features)

# %% [markdown]
# ### **8. Execute Feature Engineering Pipeline**

# %%
# Load data
print("="*60)
print("VERITASFINANCIAL - FEATURE ENGINEERING")
print("="*60)

data_loader = DataLoader(data_path='../data/processed/')
data = data_loader.load_all_data()

# Get transaction data
df_transactions = data['transactions']

print("\n" + "="*60)
print(f"TRANSACTION DATA OVERVIEW")
print("="*60)
print(f"Shape: {df_transactions.shape}")
print(f"Memory usage: {df_transactions.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"Fraud rate: {df_transactions['is_fraud'].mean()*100:.4f}%")

# Initialize and run feature engineering pipeline
pipeline = FeatureEngineeringPipeline(config={
    'amount_features': True,
    'temporal_features': True,
    'behavioral_features': True,
    'location_features': True,
    'save_intermediate': True,
    'parallel_processing': False
})

pipeline.build_pipeline()
df_features = pipeline.fit_transform(df_transactions)

# Get feature importance ranking
feature_importance = pipeline.get_feature_importance_ranking(df_features)

# %% [markdown]
# ### **9. Feature Validation and Quality Checks**

# %%
class FeatureValidator:
    """
    Validate engineered features for quality and usability.
    
    Performs checks for:
    - Missing values
    - Cardinality
    - Variance
    - Correlation
    - Distribution stability
    """
    
    def __init__(self, df, feature_names):
        """
        Initialize feature validator.
        
        Args:
            df: DataFrame with features
            feature_names: List of feature names to validate
        """
        self.df = df
        self.feature_names = feature_names
        self.validation_results = {}
        
    def run_all_checks(self):
        """Run all validation checks"""
        print("\n" + "="*60)
        print("FEATURE VALIDATION")
        print("="*60)
        
        # 1. Missing value check
        self._check_missing_values()
        
        # 2. Cardinality check
        self._check_cardinality()
        
        # 3. Variance check
        self._check_variance()
        
        # 4. Correlation check
        self._check_correlation()
        
        # 5. Distribution check
        self._check_distributions()
        
        # 6. Data type consistency
        self._check_data_types()
        
        # Summary
        self._print_summary()
        
        return self.validation_results
    
    def _check_missing_values(self):
        """Check for missing values in features"""
        missing_counts = self.df[self.feature_names].isnull().sum()
        missing_pcts = missing_counts / len(self.df) * 100
        
        high_missing = missing_pcts[missing_pcts > 5].index.tolist()
        
        self.validation_results['missing_values'] = {
            'total_missing': missing_counts.sum(),
            'features_with_missing': len(missing_counts[missing_counts > 0]),
            'high_missing_features': high_missing,
            'missing_by_feature': missing_pcts.to_dict()
        }
        
        print(f"\n📊 Missing Value Check:")
        print(f"   Total missing values: {missing_counts.sum():,}")
        print(f"   Features with missing: {len(missing_counts[missing_counts > 0])}")
        print(f"   Features >5% missing: {len(high_missing)}")
        
    def _check_cardinality(self):
        """Check cardinality of categorical features"""
        categorical_features = self.df[self.feature_names].select_dtypes(include=['object', 'category']).columns
        
        cardinality = {}
        high_cardinality = []
        
        for feat in categorical_features:
            n_unique = self.df[feat].nunique()
            cardinality[feat] = n_unique
            if n_unique > 100:
                high_cardinality.append(feat)
        
        self.validation_results['cardinality'] = {
            'categorical_features': list(categorical_features),
            'cardinality': cardinality,
            'high_cardinality_features': high_cardinality
        }
        
        print(f"\n📊 Cardinality Check:")
        print(f"   Categorical features: {len(categorical_features)}")
        print(f"   High cardinality (>100): {len(high_cardinality)}")
        
    def _check_variance(self):
        """Check variance of numerical features"""
        numerical_features = self.df[self.feature_names].select_dtypes(include=['int64', 'float64']).columns
        
        zero_variance = []
        low_variance = []
        
        for feat in numerical_features:
            variance = self.df[feat].var()
            if variance == 0:
                zero_variance.append(feat)
            elif variance < 0.01:
                low_variance.append(feat)
        
        self.validation_results['variance'] = {
            'numerical_features': list(numerical_features),
            'zero_variance_features': zero_variance,
            'low_variance_features': low_variance
        }
        
        print(f"\n📊 Variance Check:")
        print(f"   Numerical features: {len(numerical_features)}")
        print(f"   Zero variance: {len(zero_variance)}")
        print(f"   Low variance (<0.01): {len(low_variance)}")
        
    def _check_correlation(self):
        """Check for highly correlated features"""
        numerical_features = self.df[self.feature_names].select_dtypes(include=['int64', 'float64']).columns
        
        if len(numerical_features) > 1:
            corr_matrix = self.df[numerical_features].corr().abs()
            upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
            
            high_corr_pairs = []
            for col in upper_tri.columns:
                high_corr = upper_tri[col][upper_tri[col] > 0.95].index.tolist()
                for high_col in high_corr:
                    high_corr_pairs.append((col, high_col, upper_tri[col][high_col]))
            
            self.validation_results['correlation'] = {
                'high_correlation_pairs': high_corr_pairs,
                'num_high_corr_pairs': len(high_corr_pairs)
            }
            
            print(f"\n📊 Correlation Check:")
            print(f"   Highly correlated pairs (>0.95): {len(high_corr_pairs)}")
        else:
            self.validation_results['correlation'] = {'high_correlation_pairs': [], 'num_high_corr_pairs': 0}
    
    def _check_distributions(self):
        """Check distribution characteristics"""
        numerical_features = self.df[self.feature_names].select_dtypes(include=['int64', 'float64']).columns
        
        distribution_stats = {}
        skewed_features = []
        
        for feat in numerical_features[:20]:  # Limit for performance
            stats = {
                'mean': self.df[feat].mean(),
                'std': self.df[feat].std(),
                'skewness': skew(self.df[feat].dropna()),
                'kurtosis': kurtosis(self.df[feat].dropna()),
                'min': self.df[feat].min(),
                'max': self.df[feat].max()
            }
            distribution_stats[feat] = stats
            
            if abs(stats['skewness']) > 2:
                skewed_features.append(feat)
        
        self.validation_results['distributions'] = {
            'distribution_stats': distribution_stats,
            'highly_skewed_features': skewed_features
        }
        
        print(f"\n📊 Distribution Check:")
        print(f"   Highly skewed features: {len(skewed_features)}")
    
    def _check_data_types(self):
        """Check data type consistency"""
        dtypes = self.df[self.feature_names].dtypes.value_counts()
        
        self.validation_results['data_types'] = {
            'dtype_counts': dtypes.to_dict()
        }
        
        print(f"\n📊 Data Type Check:")
        for dtype, count in dtypes.items():
            print(f"   {dtype}: {count} features")
    
    def _print_summary(self):
        """Print validation summary"""
        print("\n" + "="*60)
        print("VALIDATION SUMMARY")
        print("="*60)
        
        issues = []
        
        if self.validation_results['missing_values']['high_missing_features']:
            issues.append(f"⚠ {len(self.validation_results['missing_values']['high_missing_features'])} features with >5% missing")
            
        if self.validation_results['variance']['zero_variance_features']:
            issues.append(f"⚠ {len(self.validation_results['variance']['zero_variance_features'])} features with zero variance")
            
        if self.validation_results['correlation']['num_high_corr_pairs'] > 0:
            issues.append(f"⚠ {self.validation_results['correlation']['num_high_corr_pairs']} highly correlated feature pairs")
            
        if self.validation_results['cardinality']['high_cardinality_features']:
            issues.append(f"⚠ {len(self.validation_results['cardinality']['high_cardinality_features'])} high cardinality features")
        
        if issues:
            print("\nPotential Issues Found:")
            for issue in issues:
                print(f"   {issue}")
        else:
            print("\n✅ No major issues found - features look good!")

# Run validation
validator = FeatureValidator(df_features, pipeline.feature_names)
validation_results = validator.run_all_checks()

# %% [markdown]
# ### **10. Save Engineered Features**

# %%
# Save features to disk
print("\n" + "="*60)
print("SAVING ENGINEERED FEATURES")
print("="*60)

# Create feature store directory
import os
os.makedirs('../data/features', exist_ok=True)

# Save full feature set
output_path = '../data/features/engineered_features.parquet'
df_features.to_parquet(output_path, index=False)
print(f"✅ Full feature set saved to: {output_path}")
print(f"   Shape: {df_features.shape}")
print(f"   Size: {df_features.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Save feature metadata
feature_metadata = pd.DataFrame({
    'feature_name': pipeline.feature_names,
    'data_type': [df_features[f].dtype for f in pipeline.feature_names],
    'missing_rate': [df_features[f].isnull().mean() for f in pipeline.feature_names],
    'unique_values': [df_features[f].nunique() for f in pipeline.feature_names]
})

metadata_path = '../data/features/feature_metadata.csv'
feature_metadata.to_csv(metadata_path, index=False)
print(f"✅ Feature metadata saved to: {metadata_path}")

# Save feature importance ranking
importance_df = pd.DataFrame(
    list(feature_importance.items()), 
    columns=['feature', 'correlation_with_target']
)
importance_df = importance_df.sort_values('correlation_with_target', ascending=False)

importance_path = '../data/features/feature_importance.csv'
importance_df.to_csv(importance_path, index=False)
print(f"✅ Feature importance saved to: {importance_path}")

# %% [markdown]
# ### **11. Feature Visualization**

# %%
# Visualize top features
print("\n" + "="*60)
print("FEATURE VISUALIZATION")
print("="*60)

# Create visualizations directory
os.makedirs('../reports/figures', exist_ok=True)

# 1. Top 20 features by correlation with target
top_features = importance_df.head(20)['feature'].tolist()

fig, axes = plt.subplots(5, 4, figsize=(20, 25))
axes = axes.flatten()

for i, feature in enumerate(top_features):
    if i < len(axes):
        # Split by fraud/non-fraud
        fraud_data = df_features[df_features['is_fraud'] == 1][feature]
        non_fraud_data = df_features[df_features['is_fraud'] == 0][feature]
        
        # Plot distributions
        axes[i].hist(non_fraud_data, bins=50, alpha=0.5, label='Non-Fraud', density=True)
        axes[i].hist(fraud_data, bins=50, alpha=0.5, label='Fraud', density=True)
        axes[i].set_title(f'{feature[:30]}...' if len(feature) > 30 else feature)
        axes[i].legend()

plt.tight_layout()
plt.savefig('../reports/figures/top_features_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

# 2. Feature correlation heatmap
top_20_features = importance_df.head(20)['feature'].tolist()
corr_matrix = df_features[top_20_features + ['is_fraud']].corr()

plt.figure(figsize=(15, 12))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('Feature Correlation Heatmap (Top 20 Features)')
plt.tight_layout()
plt.savefig('../reports/figures/feature_correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

# %% [markdown]
# ### **12. Summary and Next Steps**

# %%
print("\n" + "="*60)
print("FEATURE ENGINEERING COMPLETE - SUMMARY")
print("="*60)
print(f"""
✅ Successfully created {len(pipeline.feature_names)} new features across:
   • Amount features: {len([f for f in pipeline.feature_names if 'amount' in f])}
   • Temporal features: {len([f for f in pipeline.feature_names if any(x in f for x in ['hour', 'day', 'month', 'time'])])}
   • Behavioral features: {len([f for f in pipeline.feature_names if any(x in f for x in ['velocity', 'spend', 'category', 'deviation'])])}
   • Location features: {len([f for f in pipeline.feature_names if any(x in f for x in ['country', 'city', 'distance', 'ip'])])}

📊 Feature Quality:
   • Missing values: {validation_results['missing_values']['total_missing']:,} total
   • Zero variance features: {len(validation_results['variance']['zero_variance_features'])}
   • High cardinality features: {len(validation_results['cardinality']['high_cardinality_features'])}

📈 Top 5 Most Predictive Features:
""")

for i, row in importance_df.head(5).iterrows():
    print(f"   {i+1}. {row['feature']}: {row['correlation_with_target']:.4f}")

print("""
🚀 Next Steps:
   1. Proceed to Notebook 04: Model Experimentation
   2. Use these engineered features for model training
   3. Consider feature selection/dimensionality reduction
   4. Validate feature stability over time
""")

print("\n" + "="*60)